# 14 - Resultados do P1 para o Looker e testes estatísticos
**Squad 1 · Data Quality em Tempo Real | ML em Streaming**
**Responsável:** Riuler José Silvério
**Card do Trello:** Estruturação dos resultados no Looker e documentação de insights

> Card: *"Criar os insights no Looker e estruturação dos resultados adquiridos durante a comparação dos modelos e evolução de baselines. Para modelos não supervisionados, deve ter a aplicação de testes estatísticos e avaliação dos resultados."*

### O que este notebook faz
1. Treina de novo as **duas versões finais** do Isolation Forest, com a configuração já escolhida nos notebooks `10` (baseline) e `12` (evolução). Não há nova seleção de modelo aqui.
2. Aplica **testes estatísticos** para avaliar os resultados de um modelo não supervisionado.
3. Grava **5 tabelas de resultado** na Gold de ML (Delta) e no SQL Server, que é a fonte do Looker.

### Dados
Os mesmos da baseline e da evolução: dados **em tempo real da Squad 1**. A tabela de features do `09` vem da Silver de itens, que mantém as linhas com erro marcadas por flags. O treino usa os lotes até 07/10 e o teste, os lotes de 08/10 e 15/10. O rótulo fraco (regras da etapa 1) é usado **só para avaliar**. Para o nível pedido, a Silver de pedidos (somente leitura) fornece as regras de valor total e frete.

### Testes estatísticos
| Pergunta | Teste |
|---|---|
| O modelo é melhor que o acaso? | Permutação da PR-AUC |
| Os itens com erro recebem score maior que os normais? | Mann-Whitney U (efeito = AUC-ROC) |
| A evolução é melhor que a baseline, além da variação da amostra? | Bootstrap pareado (IC 95% da diferença de PR-AUC e F1) |
| Com o mesmo volume de alertas, a evolução acerta mais? | McNemar exato |
| O % de alertas é estável entre os micro-lotes? | Qui-quadrado de homogeneidade + Spearman (% alertas × % erro por lote) |
| O score dos itens normais mudou do treino para o teste? | Kolmogorov-Smirnov |

### Saídas
| Tabela (SQL Server `squad1.` / Delta `squad1/dupla1/ml/gold/`) | Grão | Uso no Looker |
|---|---|---|
| `gold_ml_metricas_itens_pedido` | versão × métrica × nível | KPIs e barras baseline × evolução, com IC 95% |
| `gold_ml_testes_estatisticos_itens_pedido` | teste × versão | Quadro dos testes estatísticos |
| `gold_ml_metricas_por_lote_itens_pedido` | micro-lote | Linha: % alertas × % erro por lote |
| `gold_ml_recall_por_regra_itens_pedido` | regra × versão | Barras de recall por regra |
| `gold_ml_alertas_itens_pedido` | item do teste | Tabela de suspeitos (preço × típico, regra, ranking) |

In [0]:
# Credenciais via .env (nunca no código). O .env fica fora do Git (.gitignore).
import os
from dotenv import load_dotenv

CAMINHOS_ENV = [".env", "../.env", "../../estagio-empregadados-turma-2/.env"]
caminho_env = next((p for p in CAMINHOS_ENV if os.path.exists(p)), None)
if caminho_env:
    load_dotenv(caminho_env)

client_id = os.getenv("ADLS_CLIENT_ID")
tenant_id = os.getenv("ADLS_TENANT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")
storage_account = os.getenv("ADLS_STORAGE_ACCOUNT_NAME")
sql_host, sql_database = os.getenv("SQL_HOST"), os.getenv("SQL_DATABASE")
sql_user, sql_password = os.getenv("SQL_USERNAME"), os.getenv("SQL_PASSWORD")
assert all([client_id, tenant_id, client_secret, storage_account]), "Credenciais do ADLS não encontradas no .env"
assert all([sql_host, sql_database, sql_user, sql_password]), "Credenciais do SQL Server não encontradas no .env"

adls_options = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": client_id,
    "fs.azure.account.oauth2.client.secret": client_secret,
    "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token",
}
base_squad1 = f"abfss://squad1@{storage_account}.dfs.core.windows.net"
caminhos = {
    "features_itens": f"{base_squad1}/dupla1/ml/features_itens_pedido",   # entrada (notebook 09)
    "silver_pedidos": f"{base_squad1}/silver/ecommerce_pedidos",          # somente leitura (regras de valor do pedido)
    "gold_ml": f"{base_squad1}/dupla1/ml/gold",                           # saída deste notebook
}
print("Credenciais carregadas do .env e caminhos prontos.")

### Etapa 1 — Leitura

In [0]:
import numpy as np
import pandas as pd

df = (spark.read.format("delta").options(**adls_options).load(caminhos["features_itens"])
      .toPandas().sort_values(["data_lote", "lote", "id_item_pedido"]).reset_index(drop=True))
pedidos = spark.read.format("delta").options(**adls_options).load(caminhos["silver_pedidos"]).toPandas()

### Etapa 2 — As duas versões finais
Mesma configuração dos notebooks `10` e `12`:
- **Baseline:** 7 features de item e pedido, `max_features = 1,0`, `contamination = 0,09`.
- **Evolução:** `rel_preco_sku` (preço ÷ típico do SKU) + quantidade + desconto ÷ preço + valor do item, `max_features = 0,5`, `contamination = 0,08`.

As duas usam 300 árvores, `max_samples = 256` e `random_state = 42`. O preço típico é a mediana do SKU no treino, calculada só com itens que passam nas regras e SKUs com 3 ou mais itens; quando isso não existe, usa a mediana da categoria.

In [0]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score
from scipy import stats

SEMENTE = 42
df["data_lote"] = pd.to_datetime(df["data_lote"])
df["categoria"] = df["sku"].str.split("-").str[0]

def referencia_preco(base_ref, x, min_itens=3):
    """Acrescenta preco_tipico e rel_preco_sku a x, com a referência calculada em base_ref (mesma regra do notebook 12)."""
    x = x.copy()
    ref_base = base_ref[base_ref.rotulo_fraco == 0]
    g_sku, g_cat = ref_base.groupby("sku").preco_unitario, ref_base.groupby("categoria").preco_unitario
    med_sku = g_sku.median()[g_sku.size() >= min_itens]
    med_cat = g_cat.median()[g_cat.size() >= min_itens]
    preco_tipico = x.sku.map(med_sku).fillna(x.categoria.map(med_cat)).fillna(ref_base.preco_unitario.median())
    x["preco_tipico"] = preco_tipico.round(2)
    x["origem_referencia"] = np.where(x.sku.isin(med_sku.index), "sku", np.where(x.categoria.isin(med_cat.index), "categoria", "geral"))
    x["rel_preco_sku"] = x.log_preco - np.log1p(preco_tipico.clip(lower=0))
    return x

treino, teste = df[df.split == "treino"], df[df.split == "teste"]
treino, teste = referencia_preco(treino, treino), referencia_preco(treino, teste)

MODELOS = {
    "baseline": dict(feats=["log_preco", "quantidade", "razao_desconto", "log_valor_liquido_item",
                            "n_itens_pedido", "n_skus_pedido", "log_valor_liquido_pedido"], mf=1.0, c=0.09),
    "evolucao": dict(feats=["rel_preco_sku", "quantidade", "razao_desconto", "log_valor_liquido_item"], mf=0.5, c=0.08),
}
modelos = {}
for nome, cfg in MODELOS.items():
    modelos[nome] = IsolationForest(n_estimators=300, max_samples=256, max_features=cfg["mf"], contamination=cfg["c"],
                                    random_state=SEMENTE, n_jobs=-1).fit(treino[cfg["feats"]])
    for base in (treino, teste):
        base[f"score_{nome}"] = -modelos[nome].score_samples(base[cfg["feats"]])     # maior = mais anômalo
        base[f"alerta_{nome}"] = (modelos[nome].predict(base[cfg["feats"]]) == -1).astype(int)

# Evolução com o mesmo volume de alertas da baseline (para comparar o mesmo esforço de revisão)
corte = np.quantile(teste.score_evolucao, 1 - teste.alerta_baseline.mean())
teste["alerta_evolucao_mesmo_volume"] = (teste.score_evolucao > corte).astype(int)

y = teste.rotulo_fraco.values
print(f"Treino: {len(treino):,} itens | Teste: {len(teste):,} itens ({y.mean():.2%} com rótulo fraco)")
for nome in MODELOS:
    print(f"{nome}: PR-AUC {average_precision_score(y, teste[f'score_{nome}']):.3f} | F1 {f1_score(y, teste[f'alerta_{nome}']):.3f}")

### Etapa 3 — Métricas com intervalo de confiança
O IC 95% vem de um **bootstrap** de 1.000 reamostragens dos itens do teste. No nível pedido, o score do pedido é o maior score entre os itens, e o pedido é anômalo se tem item com regra quebrada ou valor total/frete inválido na Silver de pedidos.

In [0]:
B = 1000
rng = np.random.default_rng(SEMENTE)
idx_boot = [rng.integers(0, len(y), len(y)) for _ in range(B)]
idx_boot = [i for i in idx_boot if 0 < y[i].sum() < len(i)]

def metricas_item(alerta, score):
    return {"pct_alertados": alerta.mean() * 100, "precision": precision_score(y, alerta, zero_division=0),
            "recall": recall_score(y, alerta), "f1": f1_score(y, alerta), "pr_auc": average_precision_score(y, score)}

def ic(alerta, score):
    reams = pd.DataFrame([{"precision": precision_score(y[i], alerta[i], zero_division=0), "recall": recall_score(y[i], alerta[i]),
                           "f1": f1_score(y[i], alerta[i]), "pr_auc": average_precision_score(y[i], score[i])} for i in idx_boot])
    return reams.quantile(0.025), reams.quantile(0.975)

linhas = []
for versao, col_alerta, col_score in [("baseline", "alerta_baseline", "score_baseline"), ("evolucao", "alerta_evolucao", "score_evolucao"),
                                      ("evolucao_mesmo_volume", "alerta_evolucao_mesmo_volume", "score_evolucao")]:
    a, s = teste[col_alerta].values, teste[col_score].values
    m = metricas_item(a, s); lo, hi = ic(a, s)
    for k, v in m.items():
        linhas.append({"nivel": "item", "versao": versao, "metrica": k, "valor": float(v),
                       "ic_inferior": float(lo.get(k, np.nan)), "ic_superior": float(hi.get(k, np.nan))})

# Nível pedido
pedidos = pedidos.drop_duplicates("id_pedido")
pedidos["rotulo_valor_pedido"] = (pedidos.dq_valor_total_invalido | pedidos.dq_frete_invalido).astype(int)
ped = (teste.groupby("id_pedido").agg(rotulo_itens=("rotulo_fraco", "max"), score_baseline=("score_baseline", "max"),
                                      score_evolucao=("score_evolucao", "max"), alerta_baseline=("alerta_baseline", "max"),
                                      alerta_evolucao=("alerta_evolucao", "max")).reset_index()
       .merge(pedidos[["id_pedido", "rotulo_valor_pedido"]], on="id_pedido", how="left").fillna({"rotulo_valor_pedido": 0}))
ped["rotulo_p1"] = ((ped.rotulo_itens == 1) | (ped.rotulo_valor_pedido == 1)).astype(int)
for versao in MODELOS:
    for rot in ["rotulo_p1", "rotulo_valor_pedido"]:
        linhas.append({"nivel": "pedido_" + rot.replace("rotulo_", ""), "versao": versao, "metrica": "pr_auc",
                       "valor": float(average_precision_score(ped[rot], ped[f"score_{versao}"])), "ic_inferior": np.nan, "ic_superior": np.nan})
for nivel, base_rot in [("item", y), ("pedido_p1", ped.rotulo_p1), ("pedido_valor_pedido", ped.rotulo_valor_pedido)]:
    linhas.append({"nivel": nivel, "versao": "aleatorio", "metrica": "pr_auc", "valor": float(np.mean(base_rot)), "ic_inferior": np.nan, "ic_superior": np.nan})

gold_metricas = pd.DataFrame(linhas)
display(gold_metricas.round(3))

### Etapa 4 — Testes estatísticos
Cada linha traz a pergunta, a estatística, o p-valor, o tamanho do efeito e a conclusão em texto, prontos para o quadro do Looker. Nível de significância: 5%.

In [0]:
import re
ALFA = 0.05
testes = []
def registrar(teste_nome, pergunta, versao, estatistica, p_valor, efeito, conclusao):
    conclusao = re.sub(r"(\d)\.(\d)", r"\1,\2", conclusao)     # vírgula decimal no texto exibido no Looker
    testes.append({"teste": teste_nome, "pergunta": pergunta, "versao": versao, "estatistica": float(estatistica),
                   "p_valor": float(p_valor), "efeito": None if efeito is None else float(efeito),
                   "significativo": bool(p_valor < ALFA), "conclusao": conclusao})

# 1) Permutação: PR-AUC melhor que o acaso
N_PERM = 2000
for versao in MODELOS:
    s = teste[f"score_{versao}"].values
    obs = average_precision_score(y, s)
    perm = np.array([average_precision_score(rng.permutation(y), s) for _ in range(N_PERM)])
    p = (np.sum(perm >= obs) + 1) / (N_PERM + 1)
    registrar("permutacao_pr_auc", "O modelo é melhor que o acaso?", versao, obs, p, obs - perm.mean(),
              f"PR-AUC {obs:.3f} contra {perm.mean():.3f} do acaso (máximo em {N_PERM} sorteios: {perm.max():.3f})")

# 2) Mann-Whitney: score dos itens com erro maior que o dos normais
for versao in MODELOS:
    s = teste[f"score_{versao}"].values
    u, p = stats.mannwhitneyu(s[y == 1], s[y == 0], alternative="greater")
    auc = u / ((y == 1).sum() * (y == 0).sum())
    registrar("mann_whitney", "Itens com erro recebem score maior que os normais?", versao, u, p, auc,
              f"Um item com erro tem score maior que um normal em {auc:.1%} das comparações")

# 3) Bootstrap pareado: diferença evolução − baseline
for metrica, fn in [("pr_auc", lambda i: average_precision_score(y[i], teste.score_evolucao.values[i]) - average_precision_score(y[i], teste.score_baseline.values[i])),
                    ("f1", lambda i: f1_score(y[i], teste.alerta_evolucao.values[i]) - f1_score(y[i], teste.alerta_baseline.values[i]))]:
    difs = np.array([fn(i) for i in idx_boot])
    lo, hi = np.percentile(difs, [2.5, 97.5])
    p = 2 * min((difs <= 0).mean(), (difs >= 0).mean())
    registrar(f"bootstrap_pareado_{metrica}", "A evolução é melhor que a baseline?", "evolucao_vs_baseline", difs.mean(), max(p, 1 / len(difs)), difs.mean(),
              f"Diferença de {metrica.upper().replace('_', '-')}: {difs.mean():+.3f} (IC 95% {lo:+.3f} a {hi:+.3f})")

# 4) McNemar exato com o mesmo volume de alertas
for rotulo, descricao in [(1, "nos itens com erro (acertos = alertar)"), (0, "nos itens normais (acertos = não alertar)")]:
    m = y == rotulo
    acerto_b = teste.alerta_baseline.values[m] == rotulo
    acerto_e = teste.alerta_evolucao_mesmo_volume.values[m] == rotulo
    so_e, so_b = int((acerto_e & ~acerto_b).sum()), int((acerto_b & ~acerto_e).sum())
    p = stats.binomtest(so_e, so_e + so_b, 0.5).pvalue if so_e + so_b else 1.0
    registrar("mcnemar_" + ("erros" if rotulo else "normais"), f"Com o mesmo volume de alertas, a evolução acerta mais {descricao}?",
              "evolucao_vs_baseline", so_e - so_b, p, so_e - so_b, f"Só a evolução acerta {so_e}; só a baseline acerta {so_b}")

# 5) Estabilidade entre micro-lotes (lotes com 50+ itens)
grandes = teste.groupby("lote").filter(lambda x: len(x) >= 50)
for versao in MODELOS:
    tab = pd.crosstab(grandes.lote, grandes[f"alerta_{versao}"])
    chi, p, gl, _ = stats.chi2_contingency(tab)
    registrar("qui_quadrado_lotes", "O % de alertas é estável entre os micro-lotes?", versao, chi, p, gl,
              "Varia entre lotes" if p < ALFA else "Sem diferença significativa entre lotes")
    por_lote = grandes.groupby("lote").agg(a=(f"alerta_{versao}", "mean"), r=("rotulo_fraco", "mean"))
    rho, p2 = stats.spearmanr(por_lote.a, por_lote.r)
    registrar("spearman_alerta_erro", "O % de alertas acompanha o % de erro real de cada lote?", versao, rho, p2, rho,
              f"Correlação de {rho:.2f} entre % de alertas e % de erro por lote")

# 6) Mudança do score dos itens normais entre treino e teste
for versao in MODELOS:
    s_tr = treino.loc[treino.rotulo_fraco == 0, f"score_{versao}"]
    s_te = teste.loc[teste.rotulo_fraco == 0, f"score_{versao}"]
    d, p = stats.ks_2samp(s_tr, s_te)
    registrar("kolmogorov_smirnov", "O score dos itens normais mudou do treino para o teste?", versao, d, p, d,
              f"D = {d:.3f}; mediana {np.median(s_tr):.3f} no treino e {np.median(s_te):.3f} no teste")

gold_testes = pd.DataFrame(testes)
display(gold_testes[["teste", "versao", "estatistica", "p_valor", "significativo", "conclusao"]].round(4))

### Etapa 5 — Métricas por micro-lote, recall por regra e itens alertados

In [0]:
REGRAS = {"quantidade 0": teste.quantidade < 1, "preço ≤ 0": teste.preco_unitario <= 0,
          "desconto > preço": teste.desconto_aplicado > teste.preco_unitario, "desconto negativo": teste.desconto_aplicado < 0}
teste["regra"] = np.select(list(REGRAS.values()), list(REGRAS.keys()), "nenhuma")

gold_por_lote = (teste.groupby(["lote", "data_lote"])
                 .agg(itens=("id_item_pedido", "size"), itens_com_erro=("rotulo_fraco", "sum"),
                      alertas_baseline=("alerta_baseline", "sum"), alertas_evolucao=("alerta_evolucao", "sum"))
                 .reset_index())
for col in ["itens_com_erro", "alertas_baseline", "alertas_evolucao"]:
    gold_por_lote["pct_" + col] = gold_por_lote[col] / gold_por_lote["itens"] * 100

gold_recall_regra = pd.DataFrame([
    {"regra": regra, "itens_com_falha": int(m.sum()), "versao": versao, "itens_alertados": int(teste.loc[m, f"alerta_{versao}"].sum()),
     "recall": float(teste.loc[m, f"alerta_{versao}"].mean()) if m.any() else np.nan}
    for regra, m in REGRAS.items() for versao in MODELOS])

alertas = teste.copy()
alertas["vezes_tipico"] = ((1 + alertas.preco_unitario) / (1 + alertas.preco_tipico)).round(2)
alertas["rank_baseline"] = alertas.score_baseline.rank(ascending=False, method="min").astype(int)
alertas["rank_evolucao"] = alertas.score_evolucao.rank(ascending=False, method="min").astype(int)
alertas["situacao"] = np.select(
    [(alertas.alerta_evolucao == 1) & (alertas.rotulo_fraco == 1), (alertas.alerta_evolucao == 1) & (alertas.rotulo_fraco == 0),
     (alertas.alerta_evolucao == 0) & (alertas.rotulo_fraco == 1)],
    ["alerta com regra quebrada", "alerta sem regra (para revisar)", "erro não alertado"], "normal")
gold_alertas = alertas[["id_item_pedido", "id_pedido", "sku", "categoria", "lote", "data_lote", "quantidade", "preco_unitario",
                        "desconto_aplicado", "preco_tipico", "origem_referencia", "vezes_tipico", "regra", "rotulo_fraco",
                        "score_baseline", "alerta_baseline", "rank_baseline", "score_evolucao", "alerta_evolucao", "rank_evolucao",
                        "situacao"]].reset_index(drop=True)

display(gold_por_lote.round(2).head())
display(gold_recall_regra.round(3))
display(gold_alertas.situacao.value_counts().rename_axis("situacao").reset_index(name="itens"))

### Etapa 6 — Gravação: Gold de ML (Delta) e SQL Server
Cada tabela é gravada em `squad1/dupla1/ml/gold/<nome>` (Delta) e em `squad1.gold_ml_<nome>_itens_pedido` (SQL Server), em `overwrite`, como no notebook `07`. Depois a tabela do SQL é relida para conferir a contagem. As tabelas são derivadas e recalculadas a cada execução.

In [0]:
from datetime import datetime
from pyspark.sql import functions as F

gerado_em = datetime.now()
TABELAS = {"metricas": gold_metricas, "testes_estatisticos": gold_testes, "metricas_por_lote": gold_por_lote,
           "recall_por_regra": gold_recall_regra, "alertas": gold_alertas}

def opcoes_sql(nome_tabela):
    return {"host": sql_host, "port": "1433", "user": sql_user, "password": sql_password,
            "database": sql_database, "dbtable": nome_tabela}

for nome, tabela in TABELAS.items():
    sdf = spark.createDataFrame(tabela.assign(gerado_em=gerado_em))
    (sdf.write.format("delta").options(**adls_options).mode("overwrite").option("overwriteSchema", "true")
        .save(f"{caminhos['gold_ml']}/{nome}"))
    nome_sql = f"squad1.gold_ml_{nome}_itens_pedido"
    sdf.write.format("sqlserver").options(**opcoes_sql(nome_sql)).mode("overwrite").save()
    gravado = spark.read.format("sqlserver").options(**opcoes_sql(nome_sql)).load().count()
    assert gravado == len(tabela), f"{nome_sql}: {gravado} linhas no SQL, esperado {len(tabela)}"
    print(f"[OK] {nome_sql}: {gravado:,} linhas | Delta em {caminhos['gold_ml']}/{nome}")

### Etapa 7 — Insights e avaliação dos resultados

> Execução de 25/11: treino com lotes até 07/10 (8.075 itens) e teste com os lotes de 08/10 e 15/10 (6.121 itens, 8,35% com rótulo fraco). Os dados são os de tempo real da Squad 1.

**Resultado no teste (item, IC 95% por bootstrap)**
| Versão | % alertados | Precision | Recall | F1 | PR-AUC |
|---|---|---|---|---|---|
| Baseline | 6,5% | 0,754 (0,713–0,795) | 0,587 (0,546–0,629) | 0,660 (0,625–0,695) | 0,741 (0,706–0,777) |
| **Evolução** | 7,9% | 0,742 (0,703–0,779) | **0,705** (0,667–0,746) | **0,723** (0,690–0,754) | **0,808** (0,773–0,840) |
| Evolução, com o mesmo volume da baseline | 6,5% | **0,809** (0,771–0,847) | **0,630** (0,589–0,671) | **0,708** (0,676–0,740) | 0,808 |

**Testes estatísticos (α = 5%)**
| Pergunta | Resultado |
|---|---|
| O modelo é melhor que o acaso? | **Sim, os dois** (permutação, p < 0,001). O melhor de 2.000 sorteios aleatórios chega a 0,10 de PR-AUC |
| Itens com erro recebem score maior? | **Sim** (Mann-Whitney, p ≈ 0). Um item com erro supera um normal em 96,9% (baseline) e 98,2% (evolução) das comparações |
| A evolução é melhor que a baseline? | **Sim.** PR-AUC +0,066 (IC 95% +0,042 a +0,089) e F1 +0,063 (IC +0,035 a +0,090); nenhum intervalo inclui o zero |
| Com o mesmo volume de alertas, acerta mais? | **Sim** (McNemar). Pega 43 erros que a baseline perde, contra 21 (p = 0,008), e dá menos falsos alarmes (42 contra 20, p = 0,007) |
| O % de alertas é estável entre lotes? | **Evolução: sim** (qui², p = 0,18). Baseline: varia (p = 0,047) |
| O % de alertas acompanha o erro real do lote? | **Sim** (Spearman 0,74 na evolução e 0,69 na baseline, p < 0,001) |
| O score dos itens normais mudou do treino para o teste? | **Um pouco** (KS, D = 0,11 e 0,17). Com milhares de itens, até uma mudança pequena é significativa; as medianas são próximas |

**Insights**
1. **Os dois modelos funcionam sem rótulo,** e não por acaso: estão quase 10 vezes acima do aleatório em PR-AUC, e o score separa os itens com erro em mais de 96% dos pares.
2. **A evolução é estatisticamente melhor** que a baseline, tanto na ordenação (PR-AUC) quanto na decisão (F1 e McNemar). Com o mesmo esforço de revisão, ela pega mais erros **e** gera menos falsos alarmes.
3. **O ganho vem de comparar o preço com o do próprio produto:** o recall em preço ≤ 0 sobe de 56% para 97%, e em desconto negativo, de 75% para 100%. Quantidade 0 continua baixa (26%), porque o zero fica colado no 1, e a regra fixa segue necessária.
4. **Os alertas acompanham o erro real de cada micro-lote,** e na evolução o volume é estável entre lotes. Isso indica que o modelo serve para pontuar o streaming.
5. **Ponto de atenção:** a distribuição do score dos itens normais muda um pouco do treino para o teste, mais na evolução. Isso é coerente com os 42% de itens do teste cujo SKU não apareceu no treino e usam a referência da categoria. Recomendação: **monitorar o score por lote e recalcular a referência de preço a cada retreino**.
6. **Para revisar:** a evolução gera 125 alertas sem regra quebrada no teste (tabela `gold_ml_alertas_itens_pedido`, situação "alerta sem regra"). Os de "× típico" alto são candidatos a uma nova regra: "preço acima de N vezes o típico do SKU".

**Limitações:** o teste tem 2 dias de lote e o rótulo fraco não é a verdade absoluta. Os intervalos de confiança refletem a variação da amostra de itens, mas não uma mudança futura nos dados.
